# SQL Analysis — Olist E-commerce
Using SQLite to query Olist data with pure SQL.
Same questions as Python — different tool, same answers.
This file contains 5 business questions answered in SQL.

In [1]:
import sqlite3
import pandas as pd

# Path to your datasets folder
path = r"E:\NILE TECH\DataAnalyst90Days\Phase1_Olist\datasets\\"

# Create a new SQLite database file
conn = sqlite3.connect(r"E:\NILE TECH\DataAnalyst90Days\Phase1_Olist\olist.db")

# Load each CSV into the database as a table
tables = {
    'orders'        : 'olist_orders_dataset.csv',
    'order_items'   : 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews' : 'olist_order_reviews_dataset.csv',
    'customers'     : 'olist_customers_dataset.csv',
    'sellers'       : 'olist_sellers_dataset.csv',
    'products'      : 'olist_products_dataset.csv',
    'translations'  : 'product_category_name_translation.csv'
}

for table_name, filename in tables.items():
    df = pd.read_csv(path + filename)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"✓ {table_name} loaded — {len(df):,} rows")

print("\nDatabase ready! All tables loaded.")

✓ orders loaded — 99,441 rows
✓ order_items loaded — 112,650 rows
✓ order_payments loaded — 103,886 rows
✓ order_reviews loaded — 99,224 rows
✓ customers loaded — 99,441 rows
✓ sellers loaded — 3,095 rows
✓ products loaded — 32,951 rows
✓ translations loaded — 71 rows

Database ready! All tables loaded.


In [2]:
def run_sql(query):
    """Run a SQL query and return results as a pandas DataFrame"""
    return pd.read_sql_query(query, conn)

print("Helper function ready. Use run_sql('YOUR QUERY HERE') to run SQL.")

Helper function ready. Use run_sql('YOUR QUERY HERE') to run SQL.


Query 1 — how many orders exist and what statuses?

In [3]:
## Count orders by status — your first GROUP BY
run_sql("""
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 1) AS percentage
FROM orders
GROUP BY order_status
ORDER BY order_count DESC
""")

,order_status,order_count,percentage
0,delivered,96478,97.0
1,shipped,1107,1.1
2,canceled,625,0.6
3,unavailable,609,0.6
4,invoiced,314,0.3
5,processing,301,0.3
6,created,5,0.0
7,approved,2,0.0


Query 2 — top 10 product categories by order volume

In [4]:
## Same question you answered in Python with groupby()
## Now answering it in SQL with JOIN + GROUP BY
run_sql("""
SELECT
    t.product_category_name_english AS category,
    COUNT(oi.order_id) AS order_count,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(AVG(oi.price), 2) AS avg_price
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN translations t ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY order_count DESC
LIMIT 10
""")

,category,order_count,total_revenue,avg_price
0,bed_bath_table,11115,1036988.68,93.30
1,health_beauty,9670,1258681.34,130.16
2,sports_leisure,8641,988048.97,114.34
3,furniture_decor,8334,729762.49,87.56
4,computers_accessories,7827,911954.32,116.51
5,housewares,6964,632248.66,90.79
6,watches_gifts,5991,1205005.68,201.14
7,telephony,4545,323667.53,71.21
8,garden_tools,4347,485256.46,111.63
9,auto,4235,592720.11,139.96


Query 3 — which sellers have the most orders?

In [5]:
run_sql("""
SELECT
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(oi.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN sellers s ON oi.seller_id = s.seller_id
GROUP BY oi.seller_id
ORDER BY total_orders DESC
LIMIT 10
""")

,seller_id,seller_city,seller_state,total_orders,total_revenue
0,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,2033,123304.83
1,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1987,200472.92
2,1f50f920176fa81dab994f9023523100,sao jose do rio preto,SP,1931,106939.21
3,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,1775,104288.42
4,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1551,160236.57
5,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1499,135171.70
6,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,1428,138968.55
7,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,1364,187923.89
8,ea8482cd71df3c1969d7b9473ff13abc,sao paulo,SP,1203,37177.52
9,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1171,141745.53


Query 4 — payment method breakdown

In [6]:
run_sql("""
SELECT
    payment_type,
    COUNT(*) AS transaction_count,
    ROUND(AVG(payment_value), 2) AS avg_payment,
    ROUND(SUM(payment_value), 2) AS total_revenue
FROM order_payments
GROUP BY payment_type
ORDER BY transaction_count DESC
""")


,payment_type,transaction_count,avg_payment,total_revenue
0,credit_card,76795,163.32,12542084.19
1,boleto,19784,145.03,2869361.27
2,voucher,5775,65.70,379436.87
3,debit_card,1529,142.57,217989.79
4,not_defined,3,0.00,0.00


Query 5 — revenue by Brazilian state

In [7]:
run_sql("""
SELECT
    c.customer_state AS state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY state
ORDER BY total_revenue DESC
LIMIT 10
""")

,state,total_orders,total_revenue
0,SP,40501,5067633.16
1,RJ,12350,1759651.13
2,MG,11354,1552481.83
3,RS,5345,728897.47
4,PR,4923,666063.51
5,SC,3546,507012.13
6,BA,3256,493584.14
7,DF,2080,296498.41
8,GO,1957,282836.70
9,ES,1995,268643.45


## Day 9 — JOINs and delivery performance analysis
Answering Business Question 1 entirely in SQL:
Which product categories have the worst delivery delays?
Which sellers should be flagged as at-risk?

In [8]:
run_sql("""
SELECT
    t.product_category_name_english AS category,
    COUNT(o.order_id) AS total_orders,
    ROUND(AVG(
        JULIANDAY(o.order_delivered_customer_date) -
        JULIANDAY(o.order_estimated_delivery_date)
    ), 1) AS avg_delay_days,
    SUM(CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 1 ELSE 0
    END) AS late_orders,
    ROUND(SUM(CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 1 ELSE 0
    END) * 100.0 / COUNT(o.order_id), 1) AS late_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN translations t ON p.product_category_name = t.product_category_name
WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY category
HAVING COUNT(o.order_id) >= 100
ORDER BY avg_delay_days DESC
LIMIT 15
""")

,category,total_orders,avg_delay_days,late_orders,late_pct
0,home_confort,429,-9.1,44,10.3
1,food,499,-9.2,49,9.8
2,audio,362,-9.5,46,12.7
3,fashion_underwear_beach,127,-10.2,16,12.6
4,electronics,2729,-10.4,266,9.7
5,construction_tools_lights,301,-10.5,30,10.0
6,books_technical,263,-10.6,29,11.0
7,telephony,4430,-10.7,369,8.3
8,auto,4139,-10.7,343,8.3
9,musical_instruments,651,-10.8,56,8.6


In [9]:
run_sql("""
SELECT
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 1 ELSE 0
    END) AS late_orders,
    ROUND(SUM(CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 1 ELSE 0
    END) * 100.0 / COUNT(DISTINCT o.order_id), 1) AS late_pct,
    CASE
        WHEN SUM(CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1 ELSE 0
        END) * 100.0 / COUNT(DISTINCT o.order_id) > 20
        THEN 'AT RISK'
        ELSE 'OK'
    END AS seller_status
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN sellers s ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date IS NOT NULL
GROUP BY oi.seller_id
HAVING total_orders >= 20
ORDER BY late_pct DESC
LIMIT 15
""")

,seller_id,seller_city,seller_state,total_orders,late_orders,late_pct,seller_status
0,2709af9587499e95e803a6498a5a56e9,sao paulo,SP,25,23,92.0,AT RISK
1,f76a3b1349b6df1ee875d1f3fa4340f0,sao paulo,SP,24,9,37.5,AT RISK
2,821fb029fc6e495ca4f08a35d51e53a5,sao paulo,SP,24,9,37.5,AT RISK
3,ede0c03645598cdfc63ca8237acbe73d,ribeirao preto,SP,43,16,37.2,AT RISK
4,835f0f7810c76831d6c7d24c7a646d4d,sao paulo,SP,42,15,35.7,AT RISK
5,54965bbe3e4f07ae045b90b0b8541f52,foz do iguacu,PR,73,26,35.6,AT RISK
6,2a261b5b644fa05f4f2700eb93544f2c,porto ferreira,SP,51,18,35.3,AT RISK
7,99a54764c341d5dc80b4a8fac4eba3fb,sao paulo,SP,44,15,34.1,AT RISK
8,4e5725ba188db8252977a4f0227bd462,ribeirao preto,SP,21,7,33.3,AT RISK
9,ad781527c93d00d89a11eecd9dcad7c1,sao jose do rio preto,SP,38,12,31.6,AT RISK


## Day 24 — Advanced SQL: window functions and CTEs
Window functions calculate across rows without grouping.
CTEs make complex queries readable and reusable.
These are the most tested advanced SQL concepts in
data analyst interviews.

### Window function 1 — ROW_NUMBER and RANK
Ranking sellers by revenue within each state.
ROW_NUMBER gives unique rank 1,2,3...
RANK gives same rank to ties (1,1,3...)

In [10]:
run_sql("""
WITH seller_revenue AS (
    SELECT
        s.seller_id,
        s.seller_state,
        s.seller_city,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(oi.price), 2) AS total_revenue
    FROM sellers s
    JOIN order_items oi ON s.seller_id = oi.seller_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY s.seller_id
    HAVING total_orders >= 10
)
SELECT
    seller_id,
    seller_state,
    seller_city,
    total_orders,
    total_revenue,
    ROW_NUMBER() OVER (
        PARTITION BY seller_state
        ORDER BY total_revenue DESC
    ) AS rank_in_state,
    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS overall_rank
FROM seller_revenue
ORDER BY seller_state, rank_in_state
LIMIT 20
""")

,seller_id,seller_state,seller_city,total_orders,total_revenue,rank_in_state,overall_rank
0,53243585a1d6dc2643021fd1853d8905,BA,lauro de freitas,348,217940.44,1,2
1,c72de06d72748d1a0dfb2125be43ba63,BA,guanambi,21,17522.00,2,140
2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,salvador,57,12656.33,3,213
3,d03698c2efd04a549382afa6623e27fb,BA,ilheus,18,8865.47,4,321
4,4aba391bc3b88717ce08eb11e44937b2,BA,arraial d'ajuda (porto seguro),27,7286.88,5,394
5,a3dd39f583bc80bd8c5901c95878921e,BA,salvador,34,4607.81,6,553
6,659e8466eb3ff1b0e8740d74fb7bbedd,BA,eunapolis,11,1486.80,7,977
7,d2e753bb80b7d4faa77483ed00edc8ca,BA,porto seguro,14,1320.70,8,1016
8,bbf9ad41dca6603e614efcdad7aab8c4,CE,fortaleza,14,7846.00,1,357
9,dbdd0ec73a4817971d962698f2fea022,CE,fortaleza,16,6384.00,2,444


### Window function 2 — LAG, LEAD, running total
LAG looks at the previous row value.
LEAD looks at the next row value.
Running total accumulates as you go down rows.
Used for MoM growth and cumulative revenue analysis.

In [11]:
run_sql("""
WITH monthly_rev AS (
    SELECT
        STRFTIME('%Y-%m', order_purchase_timestamp) AS month,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(price), 2) AS total_revenue
    FROM orders
    JOIN order_items USING (order_id)
    WHERE order_status = 'delivered'
    GROUP BY month
    ORDER BY month
)
SELECT
    month,
    total_orders,
    total_revenue,
    LAG(total_revenue) OVER (
        ORDER BY month
    ) AS prev_month_revenue,
    ROUND(
        (total_revenue - LAG(total_revenue) OVER (
            ORDER BY month)
        ) * 100.0 /
        LAG(total_revenue) OVER (ORDER BY month),
        1
    ) AS mom_growth_pct,
    ROUND(SUM(total_revenue) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS running_total
FROM monthly_rev
""")

,month,total_orders,total_revenue,prev_month_revenue,mom_growth_pct,running_total
0,2016-09,1,134.97,NaN,NaN,134.97
1,2016-10,265,40325.11,134.97,29777.1,40460.08
2,2016-12,1,10.90,40325.11,-100.0,40470.98
3,2017-01,750,111798.36,10.90,1025573.0,152269.34
4,2017-02,1653,234223.40,111798.36,109.5,386492.74
5,2017-03,2546,359198.85,234223.40,53.4,745691.59
6,2017-04,2303,340669.68,359198.85,-5.2,1086361.27
7,2017-05,3546,489338.25,340669.68,43.6,1575699.52
8,2017-06,3135,421923.37,489338.25,-13.8,1997622.89
9,2017-07,3872,481604.52,421923.37,14.1,2479227.41


### CTE — customer order value analysis
Using multiple CTEs chained together.
First CTE calculates per-order metrics.
Second CTE segments customers by spend level.
Final query summarises each segment.
This is the SQL equivalent of the RFM analysis
you built in Python on Day 17.

In [12]:
run_sql("""
-- CTE 1 — calculate total spend per customer
WITH customer_spend AS (
    SELECT
        c.customer_unique_id,
        c.customer_state,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(oi.price), 2) AS total_spend,
        ROUND(AVG(oi.price), 2) AS avg_order_value,
        MIN(o.order_purchase_timestamp) AS first_order,
        MAX(o.order_purchase_timestamp) AS last_order
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
),

-- CTE 2 — segment customers by spend level
customer_segments AS (
    SELECT
        customer_unique_id,
        customer_state,
        total_orders,
        total_spend,
        avg_order_value,
        CASE
            WHEN total_spend >= 500 THEN 'High Value'
            WHEN total_spend >= 200 THEN 'Mid Value'
            WHEN total_spend >= 100 THEN 'Standard'
            ELSE 'Low Value'
        END AS spend_segment
    FROM customer_spend
)

-- Final query — summarise each segment
SELECT
    spend_segment,
    COUNT(*) AS customer_count,
    ROUND(AVG(total_spend), 2) AS avg_spend,
    ROUND(AVG(total_orders), 2) AS avg_orders,
    ROUND(SUM(total_spend), 2) AS total_segment_revenue,
    ROUND(
        SUM(total_spend) * 100.0 /
        SUM(SUM(total_spend)) OVER (),
        1
    ) AS revenue_pct
FROM customer_segments
GROUP BY spend_segment
ORDER BY avg_spend DESC
""")

,spend_segment,customer_count,avg_spend,avg_orders,total_segment_revenue,revenue_pct
0,High Value,3634,928.76,1.11,3375120.72,25.5
1,Mid Value,11454,298.90,1.10,3423611.40,25.9
2,Standard,24804,143.81,1.04,3567008.68,27.0
3,Low Value,53466,53.41,1.01,2855757.31,21.6


### Key findings — advanced SQL

**Window functions:**
- ROW_NUMBER vs RANK — ROW_NUMBER always gives unique
  sequential numbers. RANK gives same number to ties
  then skips — so 1,1,3 not 1,1,2.
- PARTITION BY — like GROUP BY but for window functions.
  Resets the rank counter for each state.
- LAG() — accesses the previous row value. Used to
  calculate MoM growth without a self join.
- Running total — SUM() OVER (ORDER BY month ROWS BETWEEN
  UNBOUNDED PRECEDING AND CURRENT ROW) accumulates
  revenue from the first month to current month.

**CTEs:**
- WITH clause creates a named temporary result
- Multiple CTEs are chained with commas
- Each CTE can reference the previous one
- Makes complex queries readable — each step has a name
- Final SELECT reads from the last CTE

**Customer segment findings:**
- [Write your actual segment counts from the output]
- High Value customers (R$500+) are a small % but
  drive disproportionate revenue
- This mirrors the RFM Champion finding from Day 17 —
  same insight, different tool